# AgentCache — Combined Benchmark (Component 1 + 2)
**Llama-3.2-1B-Instruct · LMCache disk offload · Centroid KV injection**

Conditions measured:
- `cold` — full prefill, no tricks
- `lmcache` — LMCache NVMe offload for warm/hot prefix caching
- `centroid` — centroid KV injection skips first 64 tokens every request
- `combined` — both active (centroid domain prior + LMCache warm caching)

> **Before running:** upload the `AgentCache` repo to Google Drive at `My Drive/AgentCache`.
> Set `HF_TOKEN` in Colab Secrets for Llama-3.2 access.

## 1 — GPU check

In [ ]:
!nvidia-smi

## 2 — Install dependencies

In [ ]:
!pip install 'vllm>=0.8' lmcache peft safetensors openai tqdm transformers -q
print('done')

## 3 — Mount Drive + HF token

In [ ]:
import os, sys
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO = '/content/drive/MyDrive/AgentCache/AgentCache'  # inner AgentCache = repo root
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

os.makedirs('/content/agentcache', exist_ok=True)
os.makedirs('/content/results', exist_ok=True)
print(f'Repo: {REPO}')

## 4 — Surgically patch vLLM with centroid hooks

Adds only the centroid-specific lines to the installed files.
Does NOT replace `gpu_model_runner.py` wholesale (which would break on version mismatches).

In [ ]:
import vllm, shutil, os

VLLM = os.path.dirname(os.path.abspath(vllm.__file__))
print(f'vLLM site-packages: {VLLM}')
os.makedirs(f'{VLLM}/v1/worker/gpu', exist_ok=True)
os.makedirs(f'{VLLM}/v1/core/sched', exist_ok=True)

# Step 1: copy the two NEW files (standalone, no API conflicts)
for fname in ['centroid_injector.py', 'centroid_integration.py']:
    shutil.copy(f'{REPO}/vllm/{fname}', f'{VLLM}/{fname}')
    print(f'  copied: {fname}')

# ── gpu_model_runner.py surgical patch ────────────────────────────────────────
runner_path = f'{VLLM}/v1/worker/gpu_model_runner.py'
with open(runner_path) as f:
    src = f.read()

if 'centroid_integration' in src:
    print('  gpu_model_runner.py already patched')
else:
    ok = []

    # 1. Import block — add before the LoRA import
    IMPORT_ANCHOR = 'from vllm.lora.layers import'
    CENTROID_IMPORT = (
        'from vllm.centroid_integration import (\n'
        '    apply_centroid_block_table,\n'
        '    centroid_override_num_computed,\n'
        '    centroid_scheduler_mode,\n'
        '    ensure_centroid_injector_lazy,\n'
        '    try_load_centroid_injector,\n'
        ')\n'
    )
    if IMPORT_ANCHOR in src:
        src = src.replace(IMPORT_ANCHOR, CENTROID_IMPORT + IMPORT_ANCHOR, 1)
        ok.append('import')
    else:
        print('  WARN: could not find LoRA import anchor — centroid import NOT added')

    # 2. __init__ hook — add after layerwise_nvtx_hooks_registered line
    INIT_ANCHOR = 'self.layerwise_nvtx_hooks_registered = False'
    INIT_HOOK = (
        '\n\n        # Optional centroid KV warm-start\n'
        '        self._centroid_injector = try_load_centroid_injector(self.device)'
    )
    if INIT_ANCHOR in src:
        src = src.replace(INIT_ANCHOR, INIT_ANCHOR + INIT_HOOK, 1)
        ok.append('__init__')
    else:
        print('  WARN: could not find layerwise_nvtx_hooks_registered anchor')

    # 3. num_computed override — add after eff_num_computed = new_req_data.num_computed_tokens
    OVERRIDE_ANCHOR = 'eff_num_computed = new_req_data.num_computed_tokens'
    OVERRIDE_HOOK = (
        '\n            if (\n'
        '                not self.is_pooling_model\n'
        '                and centroid_scheduler_mode()\n'
        '                and eff_num_computed == 0\n'
        '            ):\n'
        '                ensure_centroid_injector_lazy(self)\n'
        '                eff_num_computed = centroid_override_num_computed(\n'
        '                    eff_num_computed, self._centroid_injector\n'
        '                )'
    )
    if OVERRIDE_ANCHOR in src:
        src = src.replace(OVERRIDE_ANCHOR, OVERRIDE_ANCHOR + OVERRIDE_HOOK, 1)
        ok.append('override')
    else:
        print('  WARN: could not find eff_num_computed anchor')

    # 4. KV write hook — wrap _get_block_table(0) result
    TABLE_ANCHOR = 'block_table_gid_0 = _get_block_table(0)'
    TABLE_HOOK = (
        'block_table_gid_0 = _get_block_table(0)\n'
        '        if not for_cudagraph_capture:\n'
        '            block_table_gid_0 = apply_centroid_block_table(\n'
        '                self, block_table_gid_0, num_reqs, self.input_batch,\n'
        '            )'
    )
    if TABLE_ANCHOR in src:
        src = src.replace(TABLE_ANCHOR, TABLE_HOOK, 1)
        ok.append('kv-write')
    else:
        print('  WARN: could not find block_table_gid_0 anchor')

    with open(runner_path, 'w') as f:
        f.write(src)
    print(f'  gpu_model_runner.py patched: {ok}')

# ── scheduler.py surgical patch ───────────────────────────────────────────────
# Combined-mode conflict: centroid adds 64 to num_external_computed_tokens so
# the engine skips prefill of positions 0..63 (which centroid physically wrote).
# But num_external_computed_tokens is also passed to:
#   - update_state_after_alloc  → LMCache asserts it can load that many tokens
#   - allocate_slots            → inflated value causes vLLM to mark 4 blocks as
#                                 NULL_BLOCK_ID (pending LMCache fill that never comes)
#   - can_fit_full_sequence     → memory check inflated
# All three must see _lmcache_ext (true LMCache hit count, pre-gap).
# Only num_computed_tokens (the scheduling decision) should see the inflated value.
# For allocate_slots we compensate block count by adding the gap to num_tokens
# (positional arg), so block coverage stays the same without triggering NULLs.
sched_path = f'{VLLM}/v1/core/sched/scheduler.py'
with open(sched_path) as f:
    src = f.read()

if 'centroid_sched_gap' in src:
    print('  scheduler.py already patched')
else:
    sched_ok = []

    # 1. Initialise _lmcache_ext = 0 alongside the other connector counters
    INIT_ANCHOR = 'connector_prefix_cache_queries, connector_prefix_cache_hits = 0, 0'
    if INIT_ANCHOR in src:
        src = src.replace(
            INIT_ANCHOR,
            INIT_ANCHOR + '\n                _lmcache_ext = 0',
            1,
        )
        sched_ok.append('init')
    else:
        print('  WARN: could not find connector init anchor in scheduler.py')

    # 2. After LMCache sets its hit count, snapshot it, then add centroid gap.
    #    num_external_computed_tokens is inflated here so num_computed_tokens
    #    (computed below) correctly tells the engine 64 tokens are pre-filled.
    SCHED_ANCHOR = 'connector_prefix_cache_hits = num_external_computed_tokens'
    SCHED_HOOK = (
        '\n                        _lmcache_ext = num_external_computed_tokens'
        '\n\n                    if not load_kv_async'
        ' and not self.need_mamba_block_aligned_split:\n'
        '                        from vllm.centroid_integration import centroid_sched_gap\n'
        '                        _gap = centroid_sched_gap(\n'
        '                            request.num_prompt_tokens,\n'
        '                            num_new_local_computed_tokens'
        ' + num_external_computed_tokens,\n'
        '                        )\n'
        '                        num_external_computed_tokens += _gap'
    )
    if SCHED_ANCHOR in src:
        src = src.replace(SCHED_ANCHOR, SCHED_ANCHOR + SCHED_HOOK, 1)
        sched_ok.append('gap')
    else:
        print('  WARN: could not find connector_prefix_cache_hits anchor in scheduler.py')

    # 3. update_state_after_alloc — LMCache validator must see the true hit count
    UPDATE_ANCHOR = (
        'self.connector.update_state_after_alloc(\n'
        '                        request,\n'
        '                        self.kv_cache_manager.get_blocks(request_id),\n'
        '                        num_external_computed_tokens,\n'
        '                    )'
    )
    UPDATE_FIX = (
        'self.connector.update_state_after_alloc(\n'
        '                        request,\n'
        '                        self.kv_cache_manager.get_blocks(request_id),\n'
        '                        _lmcache_ext,\n'
        '                    )'
    )
    if UPDATE_ANCHOR in src:
        src = src.replace(UPDATE_ANCHOR, UPDATE_FIX, 1)
        sched_ok.append('update')
    else:
        print('  WARN: could not find update_state_after_alloc anchor in scheduler.py')

    # 4. can_fit_full_sequence — memory check must not see centroid-inflated count
    CAN_FIT_ANCHOR = (
        'num_external_computed_tokens=num_external_computed_tokens,\n'
        '                        num_encoder_tokens=num_encoder_tokens,\n'
        '                    )\n'
        '                ):'
    )
    CAN_FIT_FIX = (
        'num_external_computed_tokens=_lmcache_ext,\n'
        '                        num_encoder_tokens=num_encoder_tokens,\n'
        '                    )\n'
        '                ):'
    )
    if CAN_FIT_ANCHOR in src:
        src = src.replace(CAN_FIT_ANCHOR, CAN_FIT_FIX, 1)
        sched_ok.append('can_fit')
    else:
        print('  WARN: could not find can_fit_full_sequence anchor in scheduler.py')

    # 5. allocate_slots — pass _lmcache_ext (not inflated) for num_external to avoid
    #    NULL_BLOCK_ID from vLLM's external-block mechanism when LMCache has no hit.
    #    Compensate block count by adding centroid gap to the positional num_tokens arg:
    #    cdiv(0 + 0 + (29+64), 16) = 6 blocks, same as before but no NULLs.
    ALLOC_SLOTS_ANCHOR = (
        '                    num_new_tokens,\n'
        '                    num_new_computed_tokens=num_new_local_computed_tokens,\n'
        '                    new_computed_blocks=new_computed_blocks,\n'
        '                    num_lookahead_tokens=effective_lookahead_tokens,\n'
        '                    num_external_computed_tokens=num_external_computed_tokens,\n'
        '                    delay_cache_blocks=load_kv_async,'
    )
    ALLOC_SLOTS_FIX = (
        '                    num_new_tokens + (num_external_computed_tokens - _lmcache_ext),\n'
        '                    num_new_computed_tokens=num_new_local_computed_tokens,\n'
        '                    new_computed_blocks=new_computed_blocks,\n'
        '                    num_lookahead_tokens=effective_lookahead_tokens,\n'
        '                    num_external_computed_tokens=_lmcache_ext,\n'
        '                    delay_cache_blocks=load_kv_async,'
    )
    if ALLOC_SLOTS_ANCHOR in src:
        src = src.replace(ALLOC_SLOTS_ANCHOR, ALLOC_SLOTS_FIX, 1)
        sched_ok.append('alloc_slots')
    else:
        print('  WARN: could not find allocate_slots anchor in scheduler.py')

    with open(sched_path, 'w') as f:
        f.write(src)
    print(f'  scheduler.py patched: {sched_ok}')

print('Patch step done.')

## 5 — Copy pre-exported centroid KV tensors from repo

The centroid files were already exported locally. No re-export needed.
If they are missing from the repo, see the note below.

In [ ]:
import shutil, numpy as np

CENTROID_K = '/content/agentcache/centroid_K.npy'
CENTROID_V = '/content/agentcache/centroid_V.npy'

# centroid_K.npy / centroid_V.npy were pre-exported on the local machine.
# Just copy them from the Drive repo — no model download required.
shutil.copy(f'{REPO}/centroid_K.npy', CENTROID_K)
shutil.copy(f'{REPO}/centroid_V.npy', CENTROID_V)
print('Copied centroid files from repo.')

K = np.load(CENTROID_K)
print(f'centroid_K shape: {K.shape}  (layers × virtual_tokens × kv_dim)')
# Expected: (16, 64, 512) — Llama-3.2-1B, 64 virtual tokens, kv_dim=8heads×64dim

## 6 — LMCache config + agent definitions

In [ ]:
import subprocess, time, json, threading, re
from dataclasses import dataclass, field, asdict
from typing import List
from openai import OpenAI
from tqdm.notebook import tqdm
import requests

MODEL = 'meta-llama/Llama-3.2-1B-Instruct'
PORT  = 8000
LMCACHE_YAML = '/content/lmcache.yaml'

# ── Centroid length cap ────────────────────────────────────────────────────────
# centroid_K.npy has 64 virtual tokens. Reduce this if output quality is low —
# fewer injected tokens = weaker prior = less distortion of model attention.
# Must be in range [1, 64]. Set to 64 to use all centroid tokens (original).
CENTROID_LEN = 32  # tokens to inject (tune this: try 16, 32, 48, 64)

with open(LMCACHE_YAML, 'w') as f:
    # chunk_size MUST be <= vllm_block_size (16) to avoid combined-mode crash.
    #
    # Root cause: with chunk_size=256 and a 93-token prompt, vLLM allocates 6
    # blocks (96 valid GPU slots). LMCache's save path builds a slot_mapping
    # spanning the full 256-token chunk. Positions 96-255 map to block column 6
    # which is NULL_BLOCK_ID (never allocated) → NULL_BLOCK_ID * 16 + offset is
    # a huge invalid index → CUDA IndexKernel crash on the decode sync.
    #
    # With chunk_size=16 (= vLLM block_size), each chunk fits in exactly one
    # block. The save slot_mapping for any chunk only accesses the column for
    # that block, which is always allocated, so no NULL_BLOCK_ID is ever reached.
    f.write('chunk_size: 16\nlocal_cpu: true\nmax_local_cpu_size: 4.0\neviction_policy: LRU\n')
print(f'LMCache config written  |  CENTROID_LEN={CENTROID_LEN}')

CODING_SYSTEM = ("""
    You are a helpful Python coding assistant. You help developers write clean, correct, and idiomatic Python code across a wide range of tasks: scripting, data processing, web services, CLI tools, testing, debugging, and system design.

When solving a task:
1. Read the request carefully before writing any code.
2. Write the minimum correct code that satisfies the request. Do not add unrequested features.
3. Prefer standard library solutions. Only use third-party packages if the user specifies them or the task clearly requires them.
4. Use idiomatic Python: list comprehensions, generators, context managers, dataclasses, and pathlib where appropriate.
5. Follow PEP 8: `snake_case` for functions and variables, `PascalCase` for classes, `UPPER_SNAKE_CASE` for module-level constants.
6. Catch specific exceptions. Never use bare `except:`. Always handle the narrowest exception type that makes sense.
7. When debugging, state your hypothesis about the root cause before proposing a fix.
8. Include a short usage example when it helps clarify the solution.
9. Add a one-line docstring to public functions. Comments should explain WHY, not WHAT.
10. Never use `eval()` or `exec()` on user-supplied input.

Code quality rules:
- Prefer `pathlib.Path` over `os.path` for filesystem operations.
- Use `with` statements for all file and resource access. Never leave file handles open.
- Avoid mutable default arguments (e.g., `def foo(items=[])` is a bug). Use `None` and initialize inside the function.
- Do not shadow built-ins: avoid naming variables `list`, `dict`, `type`, `id`, `input`, or `filter`.
- Use f-strings for string formatting. Avoid `%` formatting and `.format()` unless targeting Python < 3.6.
- Prefer `enumerate()` over manual index tracking. Prefer `zip()` over parallel index loops.
- When a function would return `None` on failure and a value on success, prefer raising an exception instead. Silent `None` returns are a common source of bugs downstream.
- Use `dataclasses.dataclass` for simple data-holding classes instead of writing `__init__`, `__repr__`, and `__eq__` manually.
- Keep functions short. If a function exceeds 40 lines, consider whether it is doing too many things.
- Avoid deep nesting. If you find yourself writing four or more levels of indentation, extract a helper function.

Type annotations:
- Annotate all public function signatures with type hints.
- Use `from __future__ import annotations` at the top of files targeting Python 3.9 or earlier to enable postponed evaluation of annotations.
- Use `Optional[X]` (or `X | None` in Python 3.10+) explicitly when a value can be None. Do not leave it implicit.
- Prefer `Sequence` over `list` and `Mapping` over `dict` in function parameters when you only need read access.
- Use `TypeVar` and `Generic` when writing reusable container or utility code. Do not over-annotate internal implementation details.

Testing guidance:
- When writing tests, use `pytest`. Avoid `unittest` unless the user requests it.
- Each test should verify one behavior. Do not write omnibus tests that check many things at once.
- Use `pytest.raises` to assert that exceptions are raised. Do not catch exceptions inside tests manually.
- Prefer `tmp_path` (a pytest fixture) for tests that touch the filesystem.
- Mock only at the boundary of your system: external HTTP calls, database connections, system clocks. Do not mock internal functions.
- Name tests descriptively: `test_parse_returns_empty_list_on_blank_input` is better than `test_parse_1`.
- Parametrize tests with `@pytest.mark.parametrize` rather than duplicating test bodies for similar inputs.
- Avoid `time.sleep` in tests. Use monkeypatching or fake clocks for time-dependent logic.

Concurrency guidance:
- Use `asyncio` for I/O-bound concurrency. Use `concurrent.futures.ThreadPoolExecutor` for blocking I/O in a sync context. Use `multiprocessing` only for CPU-bound work.
- Always `await` coroutines. Never call a coroutine without awaiting it.
- Use `asyncio.Semaphore` to cap concurrent operations. Do not fire unlimited concurrent tasks.
- In async code, prefer `asyncio.TaskGroup` (Python 3.11+) over bare `asyncio.gather` for structured concurrency with proper error propagation.
- Do not use `asyncio.get_event_loop()` in new code. Use `asyncio.run()` at the top level and pass the loop implicitly.

Error handling and logging:
- Raise `ValueError` for invalid arguments, `TypeError` for wrong types, `RuntimeError` for unexpected internal states.
- Define custom exception classes when callers need to distinguish your errors from built-in ones.
- Use the `logging` module, not `print`, for diagnostic output. Set the level at the entry point, not inside library code.
- Log at `DEBUG` for detailed traces, `INFO` for progress milestones, `WARNING` for recoverable anomalies, `ERROR` for failures that need attention. Never use `CRITICAL` unless the process cannot continue.
- Never log passwords, tokens, API keys, or personally identifiable information. Redact before logging.
- Include context in log messages: log the input that caused a failure, not just the exception class.

Security rules:
- Never construct shell commands by concatenating user input. Use `subprocess.run` with a list of arguments, never `shell=True` with user data.
- Never log passwords, tokens, or personally identifiable information. Redact before logging.
- Use the `secrets` module for generating tokens, nonces, and random identifiers. Do not use `random` for security-sensitive values.
- Validate all external input at the boundary. Do not trust file contents, environment variables, or network responses without parsing and validating them.
- Use `hashlib` with a strong algorithm (SHA-256 or better) when hashing data. Never use MD5 or SHA-1 for security purposes.
- When handling file paths from external input, resolve and validate them against an allowed base directory to prevent path traversal.

Performance guidance:
- Profile before optimizing. Do not optimize code that is not on the critical path.
- Prefer generators over lists when the full sequence is not needed at once. This reduces peak memory usage.
- Use `collections.defaultdict`, `collections.Counter`, and `itertools` utilities instead of reimplementing them.
- For repeated membership tests against a large collection, use a `set` not a `list`.
- Avoid repeated attribute lookups in tight loops. Cache `obj.method` in a local variable if called thousands of times.
- Use `functools.lru_cache` or `functools.cache` for pure functions with repeated inputs, but only when profiling confirms the overhead is worth it.

Dependency and environment management:
- Always specify package versions in requirements files. Unpinned dependencies break reproducibility.
- Use a virtual environment. Never install packages into the system Python.
- Separate production dependencies from development dependencies (e.g., `requirements.txt` vs `requirements-dev.txt`).
- Do not commit `.env` files or secrets to version control. Use environment variables and document the required keys in a `.env.example`.

When you are uncertain:
- Say so explicitly. Do not fabricate API signatures or library behavior.
- If the correct approach depends on a detail the user has not provided (Python version, framework, scale, deployment target), ask before writing code.
- If multiple valid approaches exist, briefly state the tradeoff and pick one. Do not write multiple competing implementations unless asked.
- If a question is outside Python or software engineering, say so rather than speculating.

Code review mindset:
- Before submitting code, read it as if you are the reviewer, not the author.
- Check: does every branch have an exit? Can any input cause an infinite loop or unbounded recursion?
- Check: are all resources (files, sockets, locks, subprocesses) released in all exit paths, including exceptions?
- Check: does the code handle empty input, zero-length sequences, and None arguments explicitly?
- Check: are there any magic numbers or string literals that should be named constants?
- Check: would a colleague understand this code in six months without asking the author?

Project structure guidance:
- Keep `__init__.py` files minimal. They should export the public API, not contain implementation.
- Place entry points (CLI scripts, server startup) in a separate `__main__.py` or `cli.py`, not in library modules.
- Separate I/O from logic. A function that reads a file and processes its content is harder to test than two separate functions.
- Use `if __name__ == "__main__":` guards in any script that can be imported. Code at module level runs on import, which breaks tests and tools.
- Group imports in three blocks separated by blank lines: standard library, third-party, local. Within each block, sort alphabetically.

Compatibility and versioning:
- Note the minimum Python version required by your code. Use `sys.version_info` guards only when unavoidable.
- Prefer `tomllib` (Python 3.11+) or `tomli` for reading TOML config files over custom parsers.
- Use `importlib.resources` (not `__file__` path hacks) to access data files bundled with a package.
- When deprecating a function, use `warnings.warn` with `DeprecationWarning` and a clear migration message before removing it.
- Do not use features marked as deprecated in the Python version you are targeting. Check the deprecation schedule.

Output formatting:
- When presenting data to the user, prefer structured output (tables, JSON, YAML) over ad hoc string concatenation.
- Use `pprint.pprint` for debugging nested structures. Remove it before committing.
- When writing CLI tools, send diagnostic output to `stderr` and data output to `stdout` so they can be piped independently.
- Use `argparse` for CLI argument parsing. Do not parse `sys.argv` manually.
- Format numbers with explicit precision: `f"{value:.2f}"` not `str(value)`. Floating point repr varies across platforms.

Strict behavior rule for evaluation:
Always end the final response with the exact token: GOODBYE
"""
)
SEARCH_SYSTEM = (
"""You are a helpful general search and research assistant. You help users find accurate, relevant information across a wide range of topics: science, history, current events, technology, law, medicine, culture, and more.

When handling a query:
1. Read the question carefully before searching or answering.
2. Provide the most relevant, accurate answer. Do not pad responses with unrequested information.
3. Prefer well-established, authoritative sources when synthesizing answers.
4. When the topic is time-sensitive, note whether your information may be outdated.
5. Distinguish between facts, expert consensus, and contested claims.
6. When multiple interpretations of a question exist, state them and address the most likely one.
7. When the answer is unclear or contested, say so explicitly.
8. Summarize findings concisely. Offer more detail only if asked.
9. If the user asks for sources, list them clearly.
10. Do not speculate as if it is fact. Label uncertain information as uncertain.

Search quality rules:
- Prioritize primary sources (government agencies, peer-reviewed research, official documentation) over secondary or aggregated sources.
- Do not treat popularity or repetition as evidence of accuracy.
- When a topic has active expert disagreement, represent that disagreement accurately rather than picking a side.
- Distinguish correlation from causation when presenting research findings.
- For medical, legal, and financial topics, note that the response is for informational purposes and recommend consulting a qualified professional.
- Do not present a partial result as a complete answer. If coverage is incomplete, say what is and is not covered.
- Avoid presenting outdated information as current without a date qualifier.
- When evidence is thin or emerging, say so. Do not present preliminary findings as settled science.

Citation and sourcing:
- When citing sources, include the author or organization, the title, and the publication date if known.
- Do not fabricate source names, URLs, or publication details.
- Distinguish between direct quotations and paraphrased summaries.
- When a claim is widely attributed but hard to trace to a primary source, note that.
- Prefer citing the original study or document over a news article that reports on it.
- When multiple sources conflict, surface the conflict rather than choosing one silently.
- Use consistent citation format within a single response.

Verification guidance:
- Before presenting a specific statistic or claim, consider whether it is plausible and consistent with related known facts.
- If a claim seems surprising or counterintuitive, flag it and suggest the user verify against primary sources.
- Do not cross-contaminate information from different time periods or contexts as if it applies uniformly.
- When asked about events after your knowledge cutoff, say so clearly rather than guessing.
- If a user provides a premise that appears factually incorrect, correct it gently before answering the rest of the question.

Query strategy guidance:
- For broad questions, scope the answer before diving into detail. State what angle you are covering.
- For multi-part questions, address each part in order. Do not merge distinct questions into a vague combined answer.
- When a question is ambiguous, pick the most plausible interpretation and state which one you chose.
- For comparative questions, use a consistent framework across all items being compared.
- When the answer depends heavily on geography, jurisdiction, or time period, make those qualifications explicit.

Privacy and safety rules:
- Do not assist in locating private personal information about individuals (home addresses, phone numbers, financial details).
- For queries about sensitive health, legal, or safety topics, provide general information and direct users to qualified professionals.
- Do not provide step-by-step instructions for activities that pose significant risk of physical harm.
- Treat sensitive demographic and personal information with care. Do not surface it unnecessarily.
- Do not assist with surveillance, tracking, or identifying individuals without their consent.

Handling contested and sensitive topics:
- On politically contentious topics, present the main positions accurately without advocating for one.
- On empirically contested topics (areas where scientific evidence is genuinely uncertain), reflect that uncertainty rather than overstating consensus or doubt.
- On topics where scientific consensus exists, represent that consensus clearly, even if it is politically controversial.
- Avoid false balance: not every topic has two equally valid sides.
- When a topic involves risk of harm, apply proportionate caution. Higher risk warrants more careful framing.

When you are uncertain:
- Say so explicitly. Do not fabricate facts, statistics, or source details.
- If the question depends on context the user has not provided (location, date, jurisdiction), ask before answering.
- If multiple valid answers exist depending on interpretation, briefly explain the tradeoff and state which you are addressing.
- If a question is outside your knowledge or capability, say so clearly rather than speculating.
- If you realize mid-response that you are uncertain about a claim you already made, correct yourself immediately.

Response mindset:
- Before finalizing a response, ask: is this complete? Did I miss a key aspect of the question?
- Check: did I answer what was asked, or did I answer a related but different question?
- Check: are there implicit assumptions in the question that I should surface rather than silently adopt?
- Check: would a careful reader find this response ambiguous or misleading in any part?
- Check: is everything I stated something I can stand behind, or did I hedge appropriately where needed?

Output formatting:
- Prefer structured output (bullet points, tables, numbered lists) when comparing multiple items or presenting step-by-step processes.
- Use plain prose for conversational questions and single-fact answers.
- Bold key terms when introducing them. Do not bold entire sentences.
- Keep responses proportional to the complexity of the question. A simple factual question does not need five paragraphs.
- When presenting a timeline, sort events chronologically and label dates clearly.
- Avoid filler phrases: "Great question!", "Certainly!", and "As an AI language model" add no value. Start with the answer.

Source currency and timeliness:
- Note your knowledge cutoff clearly when answering questions about fast-moving fields (AI, geopolitics, ongoing research).
- For questions about regulations, laws, or policies, note that these change frequently and recommend verifying with current official sources.
- Prefer more recent sources when both older and newer sources are available, unless the older source is the primary or foundational reference.
- Do not present a historical state of affairs as current without checking whether it has changed.

Strict behavior rule for evaluation:
Always end the final response with the exact token: GOODBYE
"""
)
AGENT_SYSTEMS = {'coding': CODING_SYSTEM, 'search': SEARCH_SYSTEM}

CODING_QUERIES = [
    'Implement a thread-safe LRU cache in Python with O(1) get and put.',
    'Write a Python context manager that retries a block up to N times on exception.',
    'Design a rate limiter class using the token bucket algorithm.',
    'Implement a trie for autocomplete with insert and search methods.',
    'Write a decorator that caches function results with a TTL.',
    'Optimize this O(n^2) solution to find pairs summing to a target value.',
    'Implement consistent hashing for a distributed cache.',
    'Write a generator that streams large CSV files without loading into memory.',
    'Implement BFS and DFS on a graph represented as an adjacency list.',
    'Design a simple pub/sub event system in Python.',
]
SEARCH_QUERIES = [
    'What are the main architectural differences between transformers and Mamba SSMs?',
    'Explain how RLHF differs from DPO in LLM fine-tuning.',
    'What is the current state of quantum computing for practical applications?',
    'Summarize the key ideas behind retrieval-augmented generation.',
    'How does PagedAttention improve GPU memory efficiency in LLM serving?',
    'What were the main contributions of the Attention is All You Need paper?',
    'Explain the difference between KV cache quantization and weight quantization.',
    'What is speculative decoding and how does it speed up inference?',
    'How does FlashAttention reduce memory usage compared to standard attention?',
    'What are the tradeoffs between beam search and sampling for text generation?',
]
PROMPTS = (
    [{'agent': 'coding', 'query': q} for q in CODING_QUERIES] +
    [{'agent': 'search', 'query': q} for q in SEARCH_QUERIES]
)
print(f'{len(PROMPTS)} prompts ({len(CODING_QUERIES)} coding, {len(SEARCH_QUERIES)} search)')

## 7 — Server + benchmark helpers

In [ ]:
@dataclass
class RequestResult:
    agent_type: str
    query_idx: int
    query: str
    ttft: float
    total_time: float
    cache_state: str
    turn_num: int = 0
    output: str = ''

@dataclass
class BenchmarkResult:
    config_name: str
    requests: List[RequestResult] = field(default_factory=list)

    def summary(self):
        by_state = {}
        for r in self.requests:
            by_state.setdefault(r.cache_state, []).append(r.ttft)
        print(f"\n{'='*45}")
        print(f'Config: {self.config_name}')
        for state in ['cold', 'warm']:
            vals = by_state.get(state, [])
            if vals:
                print(f'  {state:5s}: {sum(vals)/len(vals)*1000:7.1f} ms  (n={len(vals)})')
        print(f"{'='*45}")

    def save(self, path):
        with open(path, 'w') as f:
            json.dump({'config': self.config_name,
                       'requests': [asdict(r) for r in self.requests]}, f, indent=2)
        print(f'Saved: {path}')


def _env(use_lmcache, use_centroid):
    env = os.environ.copy()
    env['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
    if use_lmcache:
        env['LMCACHE_CONFIG_FILE'] = LMCACHE_YAML
    else:
        env.pop('LMCACHE_CONFIG_FILE', None)
    if use_centroid:
        env['VLLM_CENTROID_K_PATH']     = CENTROID_K
        env['VLLM_CENTROID_V_PATH']     = CENTROID_V
        env['VLLM_CENTROID_SCHEDULER']  = '1'
        env['VLLM_CENTROID_SYS_TOKENS'] = '0'
        env['VLLM_CENTROID_LEN']        = str(CENTROID_LEN)
    else:
        for k in ('VLLM_CENTROID_K_PATH','VLLM_CENTROID_V_PATH',
                  'VLLM_CENTROID_SCHEDULER','VLLM_CENTROID_SYS_TOKENS',
                  'VLLM_CENTROID_LEN'):
            env.pop(k, None)
    env.pop('VLLM_CENTROID_USE_LMCACHE', None)
    return env


def _cmd(use_lmcache, use_centroid=False):
    cmd = [
        sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
        '--model', MODEL, '--port', str(PORT),
        '--max-model-len', '4096',
        '--gpu-memory-utilization', '0.85',
    ]
    # APC needed for any caching condition so the growing conversation
    # prefix is reused across turns (centroid-only: GPU APC only;
    # lmcache: also offloads those blocks to CPU/disk).
    if use_lmcache or use_centroid:
        cmd.append('--enable-prefix-caching')
    if use_lmcache:
        cmd += ['--kv-offloading-backend', 'lmcache',
                '--kv-offloading-size', '4',
                '--disable-hybrid-kv-cache-manager']
    return cmd


def start_server(use_lmcache, use_centroid, label):
    logfile = f'/content/vllm_{label}.log'
    proc = subprocess.Popen(
        _cmd(use_lmcache, use_centroid), env=_env(use_lmcache, use_centroid),
        stdout=open(logfile, 'w'), stderr=subprocess.STDOUT,
    )
    print(f'vLLM starting (pid {proc.pid}) → {logfile}')
    return proc


def wait_ready(timeout=600):
    print('Waiting for vLLM...', end='')
    for i in range(timeout // 5):
        try:
            if requests.get(f'http://localhost:{PORT}/health', timeout=2).status_code == 200:
                print(' ready.')
                return True
        except Exception:
            pass
        if i % 12 == 11:
            print(f' {(i+1)*5}s...', end='')
        time.sleep(5)
    print(' TIMEOUT')
    return False


def kill_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=20)
    except subprocess.TimeoutExpired:
        proc.kill()
    time.sleep(3)
    print('vLLM stopped.')


def build_messages(system_text, history, user_text):
    """Build a growing context: system + all prior Q/A pairs + current query."""
    msgs = [{'role': 'system', 'content': system_text}]
    for user_q, asst_ans in history:
        msgs.append({'role': 'user',      'content': user_q})
        msgs.append({'role': 'assistant', 'content': asst_ans})
    msgs.append({'role': 'user', 'content': user_text})
    return msgs


def measure_ttft(client, messages, agent_type, idx, query, state, turn_num=0):
    t0 = time.perf_counter()
    first = None
    chunks = []
    try:
        for chunk in client.chat.completions.create(
            model=MODEL, messages=messages, max_tokens=128, temperature=0.0, stream=True
        ):
            if chunk.choices and chunk.choices[0].delta.content:
                token = chunk.choices[0].delta.content
                if first is None:
                    first = time.perf_counter()
                chunks.append(token)
    except Exception as e:
        print(f'    [WARN] turn={turn_num} request failed: {e}')
    total = time.perf_counter() - t0
    return RequestResult(
        agent_type=agent_type, query_idx=idx, query=query[:60],
        ttft=(first - t0) if first else total,
        total_time=total, cache_state=state,
        turn_num=turn_num,
        output=''.join(chunks),
    )


def run_benchmark(config_name, num_rounds=2):
    client = OpenAI(base_url=f'http://localhost:{PORT}/v1', api_key='none')
    result = BenchmarkResult(config_name=config_name)
    coding   = [p for p in PROMPTS if p['agent'] == 'coding']
    search   = [p for p in PROMPTS if p['agent'] == 'search']
    interleaved = [x for pair in zip(coding, search) for x in pair]

    for rnd in range(num_rounds):
        state = ['cold', 'warm'][min(rnd, 1)]
        print(f'\n--- Round {rnd+1} ({state}) ---')
        # Per-agent history reset each round so rounds replay the same token sequence
        agent_history = {'coding': [], 'search': []}
        for i, prompt in enumerate(tqdm(interleaved)):
            agent = prompt['agent']
            query = prompt['query']
            turn_num = len(agent_history[agent]) + 1
            messages = build_messages(AGENT_SYSTEMS[agent], agent_history[agent], query)
            r = measure_ttft(
                client, messages=messages,
                agent_type=agent, idx=i,
                query=query, state=state, turn_num=turn_num,
            )
            result.requests.append(r)
            agent_history[agent].append((query, r.output))
            print(f'\n  [{r.agent_type:6s}] turn={turn_num:2d}  {r.ttft*1000:6.1f}ms  ({state})')
            print(f'  Q: {query}')
            print(f'  A: {r.output}')
            print()
    return result


def run_condition(label, use_lmcache, use_centroid, num_rounds=2):
    print(f'\n{"="*60}\nCondition: {label}  (lmcache={use_lmcache}, centroid={use_centroid})\n{"="*60}')
    proc = start_server(use_lmcache, use_centroid, label)
    if not wait_ready():
        print('ERROR: check /content/vllm_{label}.log')
        kill_server(proc)
        return None
    result = run_benchmark(label, num_rounds)
    result.summary()
    result.save(f'/content/results/results_{label}.json')
    kill_server(proc)
    return result


print('helpers ready')

## 8 — Condition A: cold baseline (no LMCache, no centroid)

In [ ]:
result_cold = run_condition('cold', use_lmcache=False, use_centroid=False)

## 9 — Condition B: LMCache only (Component 1)

In [ ]:
result_lmcache = run_condition('lmcache', use_lmcache=True, use_centroid=False)

## 10 — Condition C: centroid only (Component 2)

In [ ]:
result_centroid = run_condition('centroid', use_lmcache=False, use_centroid=True)

## 11 — Condition D: combined (LMCache + centroid)

> Centroid injects virtual KV for positions 0–63 on every request.
> LMCache additionally warms positions 64–sys_len on subsequent requests
> for the same system prompt. For system prompts ≈ 64 tokens the additional
> gain over centroid-only is modest; it compounds on longer system prompts.

In [ ]:
result_combined = run_condition('combined', use_lmcache=True, use_centroid=True)

## 12 — Plot + summary

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

all_results = {
    label: res for label, res in [
        ('cold',     result_cold),
        ('lmcache',  result_lmcache),
        ('centroid', result_centroid),
        ('combined', result_combined),
    ] if res is not None
}

states  = ['cold', 'warm']
labels  = list(all_results.keys())
colors  = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

# Per-state mean TTFT
means = {}
for label, result in all_results.items():
    means[label] = [
        (lambda v: sum(v)/len(v)*1000 if v else 0)(
            [r.ttft for r in result.requests if r.cache_state == s]
        )
        for s in states
    ]

# Per-turn mean TTFT for warm round (shows caching compounding over conversation)
turn_means = {}
all_turns = sorted({r.turn_num for res in all_results.values() for r in res.requests if r.turn_num > 0})
for label, result in all_results.items():
    turn_means[label] = []
    for t in all_turns:
        vals = [r.ttft*1000 for r in result.requests if r.turn_num == t and r.cache_state == 'warm']
        turn_means[label].append(sum(vals)/len(vals) if vals else 0)

x     = np.arange(len(states))
width = 0.8 / max(len(labels), 1)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ── Left: grouped bar — Cold vs Warm ──────────────────────────────────────────
ax = axes[0]
for i, label in enumerate(labels):
    offset = (i - len(labels)/2 + 0.5) * width
    bars = ax.bar(x + offset, means[label], width, label=label, color=colors[i % len(colors)])
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 3, f'{h:.0f}', ha='center', fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(states)
ax.set_xlabel('Cache State'); ax.set_ylabel('Mean TTFT (ms)')
ax.set_title('Mean TTFT by Condition and Cache State')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# ── Middle: speedup over cold baseline ────────────────────────────────────────
ax2 = axes[1]
cold_vals = means.get('cold', [1, 1])
for i, label in enumerate(labels):
    if label == 'cold':
        continue
    speedups = [cold_vals[j] / max(means[label][j], 1) for j in range(len(states))]
    ax2.plot(states, speedups, 'o-', label=label, color=colors[i % len(colors)], linewidth=2, markersize=8)
ax2.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='cold baseline')
ax2.set_xlabel('Cache State'); ax2.set_ylabel('Speedup vs cold condition')
ax2.set_title('TTFT Speedup over Cold Baseline')
ax2.legend(); ax2.grid(alpha=0.3)

# ── Right: per-turn TTFT in warm round ────────────────────────────────────────
ax3 = axes[2]
for i, label in enumerate(labels):
    if all_turns and turn_means[label]:
        ax3.plot(all_turns, turn_means[label], 'o-', label=label,
                 color=colors[i % len(colors)], linewidth=2, markersize=6)
ax3.set_xlabel('Turn Number (within conversation)')
ax3.set_ylabel('Mean TTFT (ms) — warm round')
ax3.set_title('Per-Turn TTFT (Warm Round)\nShows caching benefit compounding over turns')
ax3.legend(); ax3.grid(alpha=0.3)
if all_turns:
    ax3.set_xticks(all_turns)

plt.tight_layout()
plt.savefig('/content/results/combined_ttft.png', dpi=150)
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print(f'\n{"="*58}')
print('AGENTCACHE COMBINED RESULTS')
print(f'{"="*58}')
print(f'{"Condition":<12} {"Cold (ms)":>12} {"Warm (ms)":>12} {"Speedup":>10}')
print('-' * 58)
cold_cold = means.get('cold', [1])[0]
for label, m in means.items():
    speedup = cold_cold / max(m[1], 0.001)
    print(f'{label:<12} {m[0]:>12.1f} {m[1]:>12.1f} {speedup:>9.2f}x')
print()
print('Note: Speedup = cold-baseline cold TTFT / condition warm TTFT')
print('      Combined: best cold (centroid) + good warm (LMCache) — each round is optimised')


## 13 — Download results

In [ ]:
from google.colab import files
for label in all_results:
    files.download(f'/content/results/results_{label}.json')
files.download('/content/results/combined_ttft.png')